# MoE Scaling Practice


In [ ]:
N_active_ref = 1_392_493_312   # P_active(ref), OLMoE scale
N_total_ref  = 7_033_078_528   # P_total(ref)

### 1. Chinchilla: compute-optimal scaling

For a fixed training compute budget $C$ (in FLOPs), what model size $N$ (parameters) and dataset size $D$ (tokens) minimize the final loss? Hoffmann et al. (2022) fit a loss of the form

$$
L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}
$$

and found $\alpha \approx \beta$, so the optimum splits extra compute evenly between model size and data:

$$
N^*(C) \propto C^{0.5}, \qquad D^*(C) \propto C^{0.5}
$$

### 2. From scaling laws to a rule of thumb: $D \approx 20N$

Training FLOPs are approximated by C = 6ND, and since $N^*$ and $D^*$ grow at the same rate, their ratio $D^*/N^*$ is constant. Empirically it comes out near

$$
\boxed{D \approx 20N}
$$

So a **1B-parameter model** is compute-optimal at roughly **20B tokens**.

> This constant was fit on dense models and depends on the setup and data quality. For MoE, $N$ must be read as *active* parameters, since $C \approx 6ND$ counts only the FLOPs actually spent per token.

In [ ]:
import math

C_budget = 1e21
N_opt = math.sqrt(C_budget / 120)
D_opt = 20 * N_opt

print(f"N_opt = {N_opt:.3e}  ({N_opt/1e9:.2f}B)")
print(f"D_opt = {D_opt:.3e}  ({D_opt/1e9:.2f}B tokens)")

N_opt = 2.887e+09  (2.89B)
D_opt = 5.774e+10  (57.74B tokens)


### Which N?

Chinchilla's derivation assumes every parameter costs the same to train *and* the same to hold in memory/serve — true for a dense model, false for MoE. C=6ND is measuring **training FLOPs**, and FLOPs only care about what's active per token.

Q2.1: Using D = 20*N_active for `ref`, what's the chinchilla-recommended token count, and what total training compute C does that imply?

In [ ]:
D_opt = 20 * N_active_ref
C_opt = 6 * N_active_ref * D_opt

D_not_flop_opt = 20 * N_total_ref
C_not_flop_opt = 6 * N_total_ref * D_not_flop_opt

print(f"FLOP-optimal (active): D={D_opt/1e9:.2f}B tokens,  C={C_opt:.3e} FLOPs")
print(f"if you used total:     D={D_not_flop_opt/1e9:.2f}B tokens,  C={C_not_flop_opt:.3e} FLOPs  ({C_not_flop_opt/C_opt:.1f}x more compute than needed)")

correct (active):  D=27.85B tokens,  C=2.327e+20 FLOPs
if you used total:  D=140.66B tokens,  C=5.936e+21 FLOPs  (25.5x too much)


### Q. But N_active isn't the whole story either!

If `N_active` were *all* that mattered, a dense 1.39B model and `ref` (1.39B active / 7.03B total) should reach the same loss for the same training compute — same FLOPs, same `N` in the formula. Empirically, that's not quite true: the extra total capacity (more distinct experts to specialize into) tends to buy MoE models somewhat *better* loss than a dense model of equal active params and equal training FLOPs.

This is the subject of a real line of research on "unified scaling laws for routed models" (Clark et al., 2022, and follow-ups) — the short version is that they fit an **effective parameter count** that sits somewhere between `N_active` and `N_total`, as a function of the sparsity ratio and `k`. We're **not** deriving that formula here — it's exactly the kind of second new topic this notebook is trying not to introduce. The takeaway to hold onto:

> `N_active` is the right thing to plug into `C = 6ND` for a *compute budget*. It is a slight *underestimate* of the model's effective capacity for a *loss* prediction. Treat published Chinchilla-for-MoE numbers as using `N_active` for compute and note that papers differ on how they handle capacity — go read the actual paper before trusting a specific multiplier.
___
Q. If two MoE models have the same `N_active` and the same training `D`, but one has a higher `num_experts` (hence higher `N_total`, lower sparsity ratio) — would you predict it to have lower or higher loss? What's the mechanism (in one sentence)?  

A: Lower loss, because as we previously saw almost all experts will have non-None gradients after each training step, and hence will learn and become useful. So, in a way, adding more experts while keeping the active_params fixed is a good thing to have because now there are more experts to choose from, while keeping the FLOPS(compute)/token same.

*Refined: the lower loss is primarily driven by **increased total model capacity and finer expert specialization**, which allows the network to store more diverse knowledge. While non-zero gradients ensure experts don't "die" during training, the true driver of better performance is having that vastly larger parameter pool to route tokens to.*



## Granularity: what do you get for a fixed active-param budget?

**Granularity** refers to how finely you slice an FFN layer's overall capacity into smaller, independent sub-experts. Higher granularity means using **smaller expert sizes** while activating **more experts simultaneously** ($k$), allowing tokens to combine a much richer mix of specialized sub-networks without changing the total active compute.

---

### The MoE Structural Knobs

* $d_{\text{model}}$ ($n_{\text{embd}}$): Hidden dimension of the transformer.
* $d_{\text{ffn\_ref}}$: Total intermediate FFN size of an equivalent dense baseline.
* $E_{\text{routed}}$ (`num_experts`): Total number of routed experts available.
* $E_{\text{shared}}$ (`num_shared_experts`): Number of always-active shared experts (e.g., DeepSeek style).
* $k$: Number of routed experts selected per token (top-$k$).
* $m$ (`mid` multiplier): Expert size factor relative to hidden dimension, where $d_{\text{expert}} = m \times d_{\text{model}}$.

---

### Key Formulas & Relationships

#### 1. Expert Granularity Factor ($g$)

When you divide a standard expert of size $d_{\text{std}}$ into $g$ smaller experts, the size per expert drops inversely while the number of active experts scales up directly:

$$d_{\text{expert}} \propto \frac{1}{g} \implies m_{\text{fine}} = \frac{m_{\text{ref}}}{g}$$

#### 2. Maintaining Constant Active Params ($\text{Params}_{\text{active}}$)

To keep active FFN compute constant across granularities, total active capacity $(k + E_{\text{shared}}) \times d_{\text{expert}}$ must remain fixed:

$$k_{\text{fine}} = g \cdot k_{\text{ref}} \quad \text{and} \quad E_{\text{shared, fine}} = g \cdot E_{\text{shared, ref}}$$

* **When $m$ goes down by $g\times$** $\to$ **$k$ and $E_{\text{shared}}$ must go up by $g\times$** (Directly proportional inverse).

#### 3. Preserving the Sparsity Ratio ($\text{Frac}_{\text{active}}$)

To maintain the same selection fraction $\frac{k}{E_{\text{routed}}}$ across configs:

$$E_{\text{routed, fine}} = g \cdot E_{\text{routed, ref}}$$

* **When $k$ increases by $g\times$** $\to$ **$E_{\text{routed}}$ must increase by $g\times$** to preserve the ratio $\frac{k}{E_{\text{routed}}}$.

#### 4. Total Model Capacity ($\text{Params}_{\text{total}}$)

Total static parameter memory scales with total experts $E_{\text{routed}}$:

$$\text{Params}_{\text{total, FFN}} \approx \left( E_{\text{routed}} + E_{\text{shared}} \right) \cdot (2 \cdot d_{\text{model}} \cdot d_{\text{expert}})$$

*(Note: Router parameters $\approx d_{\text{model}} \cdot E_{\text{routed}}$ grow with $E_{\text{routed}}$, adding a tiny linear overhead as granularity increases).*

---

### Knob Flow Summary

$$\text{Granularity } (g) \uparrow \quad \implies \quad \begin{cases}  m \text{ (Expert Size)} & \downarrow \text{ proportionally } (1/g) \\  k \text{ (Top-k Selection)} & \uparrow \text{ proportionally } (g) \\  E_{\text{shared}} \text{ (Shared Experts)} & \uparrow \text{ proportionally } (g) \\  E_{\text{routed}} \text{ (Total Experts)} & \uparrow \text{ proportionally } (g) \\  \text{Active FLOPS / Token} & \rightarrow \text{ Constant} \\  \text{Routing Flexibility} & \uparrow \text{ Higher combinatorics}  \end{cases}$$




Q. Design three configs, all with the **same `k/num_experts` ratio** (so the same fraction of the FFN is active per token) and the same active FFN params per block, but different granularity.


In [ ]:
def Params_per_expert(n_embd, moe_intermediate_size, bias=True):
    hidden = moe_intermediate_size
    return (n_embd*hidden + hidden) + (hidden*n_embd + n_embd)

def ffn_active_per_block(n_embd, num_experts, num_shared, k, moe_intermediate_size):
    pe = Params_per_expert(n_embd, moe_intermediate_size)
    routing_params = n_embd*num_experts + num_experts
    return (k + num_shared)*pe + routing_params

n_embd = 1024
configs = [
    dict(label="coarse",   num_experts=8,  num_shared_experts=1, k=1, mid=4),
    dict(label="ref-like", num_experts=32, num_shared_experts=2, k=4, mid=4),
    dict(label="fine",     num_experts=128, num_shared_experts=8, k=16, mid=1),
]

for c in configs:
    if c["num_experts"] is None:
        print(f"{c['label']:10s}  <- fill this in: pick num_experts/num_shared/k so k/num_experts "
              f"matches the 'ref-like' row (4/32 = 0.125) but with more, smaller-granularity experts")
        continue
    active = ffn_active_per_block(n_embd, c["num_experts"], c["num_shared_experts"], c["k"], n_embd * c["mid"])
    frac = c["k"] / c["num_experts"]
    print(f"{c['label']:10s}  num_experts={c['num_experts']:3d}  k={c['k']}  "
          f"active_frac={frac:.3f}  ffn_active/block={active:,}")

coarse      num_experts=  8  k=1  active_frac=0.125  ffn_active/block=16,795,656
ref-like    num_experts= 32  k=4  active_frac=0.125  ffn_active/block=50,395,168
fine        num_experts=128  k=16  active_frac=0.125  ffn_active/block=50,512,000


## 4 — Turning this into a real experiment

Q4.1: Design a tiny isoFLOP sweep you can actually run on `toy_train.py` + `data/shakespeare_char`. Fix a training compute budget `C` (small — this is CPU/single-GPU-pod scale, not `ref` scale). For 3–4 `(num_experts, k)` pairs at matched `k/num_experts` and roughly matched active params, solve `D = C / (6*N_active)` for the token budget, convert to `max_iters` given your `batch_size*block_size`, train each to completion, and plot final val loss vs. granularity.


In [ ]:
# Your isoFLOP sweep plan. Pick a small C (e.g. based on what a few thousand
# toy_train.py iterations at configs/base.py scale actually cost) and fill in:
C_toy_budget = None   # TODO, in FLOPs

sweep_configs = [
    # dict(label=..., num_experts=..., num_shared_experts=..., k=...),
]

for cfg in sweep_configs:
    pe = Params_per_expert(64)  # toy n_embd = 64
    pr = 64*cfg["num_experts"] + cfg["num_experts"]
    n_active_ffn = (cfg["k"]+cfg["num_shared_experts"])*pe + pr
    # you\'d add attention + embedding params here for the real N_active, omitted for brevity
    print(cfg["label"], "ffn_active/block:", n_active_ffn)